# Kaiku Finance - pipeline test

Every question follows the same fixed path, with **at most 2 model calls**:

1. **Route** - one small Gemini call returns a typed decision. A code guardrail can only move it towards the web.
2. **Retrieve** - code only: Exa (official domains + open web, in parallel) and/or pandas totals.
3. **Answer** - one Gemini call that sees only the evidence. Skipped for `static`.

The code, not the model, does the arithmetic, numbers the sources, tags the official ones and writes the source list.

| Section | Model calls |
|---|---|
| 1. Guardrail check on the 50 questions | 0 |
| 2. Routing, `PER_TYPE` questions per type | 5 x `PER_TYPE` |
| 3. Full answers, one per type | 4 |
| 4. Trace tool: how each answer was produced | 0 |
| 5. Thinking level: minimal vs low | 3 |
| 6. Exa settings | 0 |
| 7. Your own question, with its trace | up to 2 |

With `PER_TYPE = 2` the notebook uses 19 calls. With `PER_TYPE = 10` (all 50 questions) it uses 59.
Each section checks `BUDGET` before calling the model. Needs `GEMINI_API_KEY` and `EXA_API_KEY` in `.env`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # backend/notebook -> backend/src

import pandas as pd
from project_aethon.kaiku_agent import (ANSWER_PROMPT, DOC_PROMPT, MODEL_NAME, create_agent, format_evidence,
                                        load_transactions, retrieve_web, router_prompt)
from project_aethon.kaiku_eval import (QUESTIONS, UNANSWERABLE, ask_all, compare_exa_configs, compare_thinking,
                                       guardrail_summary, guardrail_table, remaining, results_table,
                                       routing_eval, routing_summary, side_by_side, thinking_verdict)
from project_aethon.kaiku_trace import explain, show_trace, timeline

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 250)

MODEL = "gemini-3.1-flash-lite"    # the daily quota is per model: change it here if this one is exhausted
BUDGET = 20            # model calls allowed for the whole notebook
PER_TYPE = 2           # questions per type in the routing test (10 = the full set of 50)

transactions = load_transactions()   # synthetic sample; pass a CSV path for the S-Bank dataset
agent = create_agent(model_name=MODEL, transactions=transactions)
print(f"Model: {MODEL} | thinking: router={agent.router_thinking}, answer={agent.answer_thinking}")
print(f"Planned calls: {5 * PER_TYPE + 4 + 3 + 2} of {BUDGET}")

Model: gemini-3.1-flash-lite | thinking: router=minimal, answer=minimal
Planned calls: 19 of 20


## 1. Guardrail check (0 calls)

The code rule that sends a question to Exa whatever the router says, run on all 50 questions.
A `static` question must have an empty `forced_to_web`.

In [2]:
guard = guardrail_table()
print(guardrail_summary(guard))
guard

static questions left to the router: 5/5 | web, both and document questions that can never be answered from training: 15/15


,type,question,forced_to_web,ok
0,static,What is a variable interest rate?,,True
1,static,How does compound interest work?,,True
2,static,What is an emergency fund for?,,True
3,static,What does APR mean on a loan?,,True
4,static,What is the difference between a fixed-rate and a variable-rate mortgage?,,True
5,web,What is the 12-month Euribor today?,time marker: today,True
6,web,How much is the Kela general housing allowance this year for a single perso,time marker: this year,True
7,web,What is the general VAT rate in Finland right now?,time marker: right now,True
8,web,Can my landlord in Finland raise my rent by 8% in the middle of the contrac,place or institution: finland,True
9,web,What is the capital income tax rate in Finland in 2026?,place or institution: finland,True


## 2. Routing (5 x PER_TYPE calls)

Only the router runs: one call per question. `why` is the router's own reason, and `decided_by`
shows whether the model or the guardrail made the final decision. Decisions are cached for section 3.

In [4]:
routing = routing_eval(agent, per_type=PER_TYPE, budget=BUDGET)
print(f"\ncalls used: {agent.stats['llm_calls']}/{BUDGET}")
routing_summary(routing)

[OK] static   -> static      | What is a variable interest rate?
[OK] static   -> static      | How does compound interest work?
[OK] web      -> web         | What is the 12-month Euribor today?
[OK] web      -> web         | How much is the Kela general housing allowance this year for a
[OK] data     -> data        | How much did I spend on taxis last month?
[OK] data     -> data        | Where is my money going? Split it into the three baskets.
[OK] both     -> both        | My flat in Helsinki is 80 m2. How does the rent I pay each mon
[OK] both     -> both        | How much do I pay in rent each month, and what is the average 
[OK] document -> web     doc | Write a short letter to my landlord objecting to a rent increa
[OK] document -> web     doc | Write an email to my landlord in Helsinki objecting to an 8% r

calls used: 10/20


,routed,correct,by_guardrail,mean_tokens,mean_seconds
type,,,,,
static,2,2,0,524,5.59
web,2,2,0,512,1.28
data,2,2,0,530,1.34
both,2,2,0,552,1.90
document,2,2,0,536,1.36
ALL,10,10,0,531,2.29


In [5]:
routing

,type,question,route,document,ok,decided_by,why,query,tokens,seconds
0,static,What is a variable interest rate?,static,False,True,model,definition of financial concept,NaN,520,9.58
1,static,How does compound interest work?,static,False,True,model,Explaining mathematical concept,NaN,528,1.60
2,web,What is the 12-month Euribor today?,web,False,True,model,current market interest rate,12-month Euribor rate November 2026,491,1.54
3,web,How much is the Kela general housing allowance this year for,web,False,True,model,check current benefit rates,Kela general housing allowance 2026 single person Helsinki monthly,533,1.03
4,data,How much did I spend on taxis last month?,data,False,True,model,user's own taxi spending,NaN,518,1.28
5,data,Where is my money going? Split it into the three baskets.,data,False,True,model,categorizing personal spending data,NaN,542,1.41
6,both,My flat in Helsinki is 80 m2. How does the rent I pay each m,both,False,True,model,comparing user rent data,average monthly rent 80m2 apartment Helsinki 2026,547,2.27
7,both,"How much do I pay in rent each month, and what is the averag",both,False,True,model,user spending and market data,average monthly rent in Helsinki 2026,556,1.52
8,document,Write a short letter to my landlord objecting to a rent incr,web,True,True,model,Legal requirements for rent,Finland rent increase legislation grounds for objection 2026,534,1.56
9,document,Write an email to my landlord in Helsinki objecting to an 8%,web,True,True,model,legal objection to rent,Helsinki rent increase regulations mid-contract 2026,538,1.15


## 3. Full answers, one per type (4 calls)

The first question of each type. Routing comes from the cache, so each case costs one answer call
and the `static` one costs nothing more. What to look for:

- **web**: official sources first, tagged `OFFICIAL`, numbered from 1, with the fragment used and the date.
- **both**: the flat is 80 m2, so the answer must give the full monthly figure for that size, not only the price per m2.
- **document**: the rule quoted word for word, a reference number after each quote, references at the end.

In [6]:
CASES = {kind: QUESTIONS[kind][0] for kind in QUESTIONS}
results = ask_all(agent, CASES, budget=BUDGET)
print(f"calls used: {agent.stats['llm_calls']}/{BUDGET}")
results_table(results)

calls used: 14/20


,case,route,calls,tokens,thinking,seconds,sources,cited,fragments_found,calcs,figures_grounded,quotes_verified,exa_usd,problems
0,static,static,1,520,0,9.58,0 official / 0,0 official / 0,0/0,0,0/0,-,0.000,
1,web,web,2,1414,0,22.42,2 official / 4,1 official / 2,1/2,0,1/1,-,0.014,
2,data,data,2,915,0,2.02,0 official / 0,0 official / 0,0/0,0,1/1,-,0.000,
3,both,both,2,1488,0,13.65,2 official / 4,0 official / 0,0/0,1,5/5,-,0.014,web evidence but no [WEB n] label; data evidence but no [DATA] label; calculation could not be evaluated
4,document,web (doc),2,1133,0,12.47,2 official / 2,1 official / 1,1/1,0,0/0,1/1,0.014,


In [7]:
for name, res in results.items():
    print("#" * 40, name, "#" * 40)
    print(res.answer)
    print()

######################################## static ########################################
A variable interest rate is a loan rate that fluctuates over time based on a market benchmark, such as the Euribor. Because the rate changes, your monthly interest payments can rise or fall throughout the loan term. This means your total repayment amount is not fixed and depends on market conditions. [MODEL]

Sources
[MODEL] Model's general knowledge (no external source)

######################################## web ########################################
The 12-month Euribor is 3.328%. This figure is based on the data from November 13, 2026 [WEB 1]. This value represents the rate recorded at the close of the session that day [WEB 2].

Sources
[WEB 1] OFFICIAL | reports.suomenpankki.fi | no date
        "…360) | 3.328 |"
        https://reports.suomenpankki.fi/WebForms/ReportViewerPage.aspx?output=&report=%2Ftilastot%2Fmarkkina-_ja_hallinnolliset_korot%2Feuribor_korot_today_en
[WEB 2] WEB | consul

## 4. Trace tool: how each answer was produced (0 calls)

`show_trace` prints the three steps of a question: what the router decided and why, what was retrieved,
which sources the answer cited or ignored, the calculations the app ran and the automatic checks.
`timeline` puts the seconds and tokens of every question in one table.

In [8]:
for name in ("both", "document"):
    if name in results:
        show_trace(results[name])
        print()

KAIKU TRACE  "My flat in Helsinki is 80 m2. How does the rent I pay each month compare with the ..."
1 ROUTE     -> BOTH                                   2.27s | 443 in / 104 out | thinking 0 | cached
  decided by: model
  why       : comparing user rent data
  guardrail : no override needed; this question can never be static (place or institution: helsinki)
  language  : en | router thinking level: minimal
  search    : "average monthly rent 80m2 apartment Helsinki 2026" | open web limited to the last 365 days
  data query: category=rent | 2026-10-14 to 2026-11-14 | group by month
----------------------------------------------------------------------------------------------------
2 RETRIEVE  Exa x2 in parallel + pandas on the transactions                       4.32s | Exa $0.014
  official  : 2 results
  open web  : 2 results
  [1] OFFICIAL pxdata.stat.fi             no date     Rent index (2025=100) and average rents per 
  [2] OFFICIAL stat.fi                    2026-07-16  Non-sub

In [9]:
timeline(results)

,case,route,decided_by,route_s,retrieve_s,answer_s,total_s,router_tok,answer_tok,thinking_tok,sources,cited,problems
0,static,static,model,9.58,0.00,0.00,9.58,520,0,0,0 official / 0,0,0
1,web,web,model,1.54,4.91,15.97,22.42,491,923,0,2 official / 4,2,0
2,data,data,model,1.28,0.01,0.73,2.02,518,397,0,0 official / 0,0,0
3,both,both,model,2.27,4.32,7.06,13.65,547,941,0,2 official / 4,0,3
4,document,web (doc),model,1.56,8.12,2.79,12.47,534,599,0,2 official / 2,1,0


## 5. Thinking level: minimal vs low (3 calls)

The answer step now runs with thinking `minimal`. This answers the same three questions with `low`,
on exactly the same evidence, and compares cost and the automatic quality checks.

In [10]:
THINK_CASES = {k: CASES[k] for k in ("web", "both", "document")}
thinking, by_level = compare_thinking(agent, THINK_CASES, levels=("minimal", "low"), budget=BUDGET)
print(thinking_verdict(thinking))
print(f"calls used: {agent.stats['llm_calls']}/{BUDGET}")
thinking

'minimal' vs 'low' on 3 case(s): answer tokens -16 %, answer seconds +99 %. Automatic checks WORSE: more label problems (3 vs 1); fewer verified quotes (1 vs 2). Read the answers side by side before deciding: the checks do not judge wording or usefulness.
calls used: 17/20


,case,thinking,answer_tokens,thinking_tokens,answer_seconds,words,label_problems,ungrounded_figures,cited,official_cited,fragments_matched,quotes_verified,quotes_total,no_evidence
0,web,minimal,923,0,15.97,34,0,0,2,1,1,0,0,False
1,both,minimal,941,0,7.06,80,3,0,0,0,0,0,0,False
2,document,minimal,599,0,2.79,140,0,0,1,1,1,1,1,False
3,web,low,1078,127,1.90,55,0,0,1,1,1,0,0,False
4,both,low,1059,139,1.80,60,1,0,0,0,0,0,0,False
5,document,low,793,122,9.25,190,0,0,2,2,2,2,2,False


In [11]:
side_by_side(by_level, "document")

============================== document | thinking: minimal ==============================
Subject: Objection to proposed rent increase for [Address of apartment]

Dear [Landlord's Name],

I am writing to formally object to the rent increase of [Amount of increase] proposed for my apartment at [Address of apartment], effective from [Start date].

My current lease agreement does not contain a clause regarding the adjustment of rent. According to the guidelines provided by the Finnish Competition and Consumer Authority:

"If a lease agreement does not contain a clause on the review of... increase can only be implemented, if both parties agree to this." [1]

Since I do not agree to this change, I request that you confirm in writing by [Date - 7 days from now] that my rent will remain at the current amount of [Current rent amount]. 

I look forward to resolving this matter promptly.

Best regards,

[Your Name]
[Your Phone Number]

Lähteet
[1] VIRALLINEN | kkv.fi | ei päivämäärää
    Rental

## 6. Exa settings (0 model calls)

These cells spend Exa credits only (the comparison costs about $0.35), so they can be re-run.
`official_share` is the part of the evidence that comes from official sources.
Check the expected texts in `EXA_EVAL` before trusting `hit_rate`.

In [12]:
got = retrieve_web("Kela general housing allowance amount")
print("searches:", got["searches"], "| Exa cost (USD):", got["cost"])
print(format_evidence(got["sources"]))

searches: [{'scope': 'official', 'results': 2, 'cost': 0.007}, {'scope': 'open web', 'results': 2, 'cost': 0.007}] | Exa cost (USD): 0.014
[1] OFFICIAL | kela.fi | General housing allowance | Our Services | Kela | 2010-02-01
"Housing allowance can cover up to 70% of your housing costs If your housing allowance is less than EUR 15 per month, it is not paid out. The easiest way to find out how much housing allowance you could get is to use our online calculator. ... - income and assets - housing costs for which you can get"
[2] OFFICIAL | laskurit.kela.fi | General housing allowance calculator
"## How to use the calculator With this calculator, you can estimate the amount of general housing allowance. Information marked ... and needs a particularly large amount of space on account of assistive devices or an outside caregiver, such a household"
[3] OFFICIAL | kela.fi | How do income and assets affect general housing ... | 2010-02-01
"## How is housing allowance calculated? General housing

In [13]:
compare_exa_configs()

,config,hit_rate,evidence_tokens,official_share,exa_cost_usd
0,official 2 + web 2 (default),1.00,239.33,0.83,0.08
1,official 3 only,0.83,247.50,1.00,0.04
2,open web 3 only,1.00,246.83,0.72,0.04
3,"official 2 + web 2, 150 chars",1.00,147.00,0.83,0.08
4,"official 2 + web 2, lite",1.00,237.50,0.83,0.05


## 7. Your own question, with its trace (up to 2 calls)

Asking the same question twice is free: the second answer comes from the cache.

In [14]:
QUESTION = "My flat in Helsinki is 50 m2. Is my monthly rent above the current average for that size?"

if remaining(agent, BUDGET) >= agent.calls_needed(QUESTION):
    own = explain(agent, QUESTION)
else:
    print(f"Skipped: {remaining(agent, BUDGET)} calls left.")
print(f"\ncalls used: {agent.stats['llm_calls']}/{BUDGET} | totals: {agent.stats}")

KAIKU TRACE  "My flat in Helsinki is 50 m2. Is my monthly rent above the current average for tha..."
1 ROUTE     -> BOTH                                           34.41s | 436 in / 126 out | thinking 0
  decided by: model
  why       : compare user rent average
  guardrail : no override needed; this question can never be static (time marker: current)
  language  : en | router thinking level: minimal
  search    : "average monthly rent 50m2 apartment Helsinki 2026" | open web limited to the last 30 days
  data query: category=rent | 2026-10-14 to 2026-11-14 | group by month
----------------------------------------------------------------------------------------------------
2 RETRIEVE  Exa x2 in parallel + pandas on the transactions                       2.39s | Exa $0.014
  official  : 2 results
  open web  : 2 results
  [1] OFFICIAL pxdata.stat.fi             no date     Rent index (2025=100) and average rents per 
  [2] OFFICIAL statfin.stat.fi            no date     Average rents per

In [15]:
# A question with no real answer: it should say so, with no invented figure.
if remaining(agent, BUDGET) >= agent.calls_needed(UNANSWERABLE):
    explain(agent, UNANSWERABLE)
else:
    print(f"Skipped: {remaining(agent, BUDGET)} calls left.")

Skipped: 1 calls left.
